In [1]:
import psycopg2
import csv
from datetime import datetime
from dotenv import load_dotenv
import os

# -----------------------------
# Carregar .env
# -----------------------------
env_path = r"C:\Users\zeno.filho\projetos\python_secrets_lib\.env"
load_dotenv(dotenv_path=env_path)

# -----------------------------
# Caminho do CSV
# -----------------------------
CSV_PATH = r"C:\Users\zeno.filho\Downloads\Matriz_Rodo_Eixo_V2.csv"


# -----------------------------
# Inferência de tipo
# -----------------------------
def infer_type(value):
    if value == "" or value is None:
        return "TEXT"
    try:
        int(value)
        return "INTEGER"
    except:
        pass
    try:
        float(value.replace(",", "."))
        return "DOUBLE PRECISION"
    except:
        pass
    try:
        datetime.fromisoformat(value)
        return "TIMESTAMP"
    except:
        pass
    return "TEXT"


# -----------------------------
# Inferência de schema do CSV
# -----------------------------
def infer_schema(csv_path, sample_size=1000):
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        headers = next(reader)

        col_types = ["TEXT"] * len(headers)

        priority = {
            "INTEGER": 1,
            "DOUBLE PRECISION": 2,
            "TIMESTAMP": 3,
            "TEXT": 4
        }

        for i, row in enumerate(reader):
            if i >= sample_size:
                break

            for idx, value in enumerate(row):
                inferred = infer_type(value)
                if priority[inferred] > priority[col_types[idx]]:
                    col_types[idx] = inferred

    return headers, col_types


# -----------------------------
# Criar tabela com schema
# -----------------------------
def create_table(conn, schema, table_name, columns, types):
    cur = conn.cursor()

    # cria schema se não existir
    cur.execute(f'CREATE SCHEMA IF NOT EXISTS "{schema}";')

    full_table_name = f'"{schema}"."{table_name}"'

    cols_sql = ", ".join(
        f'"{col}" {col_type}'
        for col, col_type in zip(columns, types)
    )

    sql = f"""
        CREATE TABLE IF NOT EXISTS {full_table_name} (
            {cols_sql}
        )
    """

    cur.execute(sql)
    conn.commit()
    cur.close()


# -----------------------------
# Conexão com banco
# -----------------------------
def get_db_connection():
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT", 5432)
    )


# -----------------------------
# COPY CSV → PostgreSQL
# -----------------------------
def copy_csv_to_postgres(csv_path, table_name, schema="public"):
    conn = get_db_connection()

    full_table_name = f'"{schema}"."{table_name}"'

    print(f"📥 Carregando arquivo: {csv_path}")
    print(f"📦 Destino: {schema}.{table_name}")

    columns, types = infer_schema(csv_path)
    create_table(conn, schema, table_name, columns, types)

    cur = conn.cursor()

    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        cur.copy_expert(
            sql=f"""
                COPY {full_table_name}
                FROM STDIN
                WITH (
                    FORMAT CSV,
                    HEADER TRUE,
                    DELIMITER ',',
                    ENCODING 'UTF8'
                )
            """,
            file=f
        )

    conn.commit()
    cur.close()
    conn.close()

    print("✅ Carga concluída com sucesso!")




In [3]:
print(host)

NameError: name 'host' is not defined

In [5]:
# -----------------------------
# EXECUÇÃO
# -----------------------------
if __name__ == "__main__":
    copy_csv_to_postgres(
        CSV_PATH,
        "tbl_mtx_veic_comercial",
        schema="VISUM"
    )


📥 Carregando arquivo: C:\Users\zeno.filho\Downloads\Matriz_Rodo_Eixo_V2.csv
📦 Destino: VISUM.tbl_mtx_veic_comercial
✅ Carga concluída com sucesso!
